In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score


from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

RANDOM_STATE = 42

In [24]:
# Φόρτωση του καθαρισμένου df από το 01_eda_cleaning
df = pd.read_pickle("df_clean.pkl")
print("Φορτώθηκε:", df.shape)

Φορτώθηκε: (112981, 46)


## **FEATURE ENGINEERING**

### Κατασκευή δεικτών **Altman**, όσοι δεν υπάρχουν ήδη στο dataset. Από τους 5 δείκτες υπάρχουν οι 2, άρα κατασκευάζω 3.

In [25]:
# Δείκτες Altman (1968) - μόνο όσοι δεν υπάρχουν ήδη στο dataset
# X2 = reoa και X3 = EBIT υπάρχουν ήδη

df['alt_x1'] = (df['act'] - df['lct']) / df['at']          # Κεφάλαιο κίνησης / Ενεργητικό
df['alt_x4'] = (df['csho'] * df['prcc_f']) / df['lt']      # Χρημ. αξία ιδίων κεφαλαίων / Υποχρεώσεις
df['alt_x5'] = df['sale'] / df['at']                       # Πωλήσεις / Ενεργητικό

# Διαιρέσεις με το μηδέν δίνουν άπειρο (inf): τις μετατρέπουμε σε κενές τιμές
df = df.replace([np.inf, -np.inf], np.nan)

print(df[['alt_x1', 'alt_x4', 'alt_x5']].describe())
print(df[['alt_x1', 'alt_x4', 'alt_x5']].isnull().sum())

              alt_x1         alt_x4         alt_x5
count  112590.000000  112881.000000  112590.000000
mean       -1.738350      16.394760       1.274677
std       103.819767     411.560261      39.694176
min    -25968.517094       0.000000      -0.899225
25%         0.017452       0.801850       0.429859
50%         0.191235       2.072653       0.914773
75%         0.409253       6.149207       1.477736
max         1.000000   78458.526316   13203.000000
alt_x1    391
alt_x4    100
alt_x5    391
dtype: int64


### Κατασκευή δεικτών **Beneish**. Από τους δείκτες του ο DEPI υπάρχει ήδη στο σύνολο δεδομένων και ο SGAI δεν υπολογίζεται λόγω έλλειψης πληροφοριών

In [26]:
# Ταξινόμηση: κάθε εταιρεία με τις χρονιές της στη σειρά
df = df.sort_values(['gvkey', 'fyear'])

# Ποια είναι η προηγούμενη χρονιά της ίδιας εταιρείας;
prev_year = df.groupby('gvkey')['fyear'].shift(1)

# True όταν η προηγούμενη γραμμή δεν είναι η αμέσως προηγούμενη χρονιά
gap = prev_year != df['fyear'] - 1

# Μεγέθη που χρειάζονται για την προηγούμενη χρονιά
priv_year_cols = ['rect', 'sale', 'cogs', 'act', 'ppegt', 'at',
            'dltt', 'lct', 'che', 'dlc', 'txp']

for col in priv_year_cols:
    df[col + '_priv_year'] = df.groupby('gvkey')[col].shift(1)
    df.loc[gap, col + '_priv_year'] = np.nan

In [27]:
print(df[['gvkey', 'fyear', 'sale', 'sale_priv_year']].head(10))

        gvkey  fyear      sale  sale_priv_year
4582     1004   1991   422.657             NaN
38301    1004   1997   782.123             NaN
65386    1004   2001   638.721             NaN
71748    1004   2002   606.337         638.721
77815    1004   2003   651.958         606.337
83796    1004   2004   747.848         651.958
101501   1004   2007  1384.919             NaN
107369   1004   2008  1423.976        1384.919
0        1009   1990    40.522             NaN
4583     1009   1991    36.222          40.522


In [28]:
# Beneish (1999) - DEPI υπάρχει ήδη (dpi), SGAI δεν υπολογίζεται

# Μηδενικοί παρονομαστές -> κενές τιμές, πριν τις διαιρέσεις
sale     = df['sale'].replace(0, np.nan)
sale_priv_year = df['sale_priv_year'].replace(0, np.nan)
at       = df['at'].replace(0, np.nan)
at_priv_year   = df['at_priv_year'].replace(0, np.nan)

# DSRI
df['ben_dsri'] = (df['rect'] / sale) / (df['rect_priv_year'] / sale_priv_year)

# GMI
gm = (df['sale'] - df['cogs']) / sale
gm_priv_year = (df['sale_priv_year'] - df['cogs_priv_year']) / sale_priv_year
df['ben_gmi'] = gm_priv_year / gm

# AQI
aq = 1 - (df['act'] + df['ppegt']) / at
aq_priv_year = 1 - (df['act_priv_year'] + df['ppegt_priv_year']) / at_priv_year
df['ben_aqi'] = aq / aq_priv_year
df.loc[aq_priv_year == 0, 'ben_aqi'] = 1      # Beneish (1999): μηδενικός παρονομαστής AQI -> 1

# SGI
df['ben_sgi'] = df['sale'] / sale_priv_year

# LVGI
lev = (df['dltt'] + df['lct']) / at
lev_priv_year = (df['dltt_priv_year'] + df['lct_priv_year']) / at_priv_year
df['ben_lvgi'] = lev / lev_priv_year

# TATA
d_act = df['act'] - df['act_priv_year']
d_che = df['che'] - df['che_priv_year']
d_lct = df['lct'] - df['lct_priv_year']
d_dlc = df['dlc'] - df['dlc_priv_year']
d_txp = df['txp'] - df['txp_priv_year']
df['ben_tata'] = ((d_act - d_che) - (d_lct - d_dlc - d_txp) - df['dp']) / at

df = df.replace([np.inf, -np.inf], np.nan)

In [29]:
ben_cols = ['ben_dsri', 'ben_gmi', 'ben_aqi', 'ben_sgi', 'ben_lvgi', 'ben_tata']
print(df[ben_cols].describe())
print(df[ben_cols].isnull().sum())

           ben_dsri        ben_gmi       ben_aqi       ben_sgi       ben_lvgi  \
count  85666.000000   87343.000000  9.206200e+04  88170.000000   91954.000000   
mean       2.445987      -1.314226  2.306805e+10      2.513354       4.143883   
std      107.492402     659.098585  8.189220e+12     82.453739     726.992191   
min     -622.621027 -192418.000000 -7.862387e+14    -49.285714       0.000000   
25%        0.828128       0.918875  6.678541e-01      0.959145       0.892640   
50%        0.985973       0.997752  9.962725e-01      1.085422       1.010559   
75%        1.151342       1.076271  1.270998e+00      1.268541       1.203512   
max    21440.500000   21839.451613  2.135547e+15  15055.000000  219974.829198   

           ben_tata  
count  92148.000000  
mean      -0.470916  
std       87.365751  
min   -25872.534188  
25%       -0.094018  
50%       -0.040013  
75%        0.004515  
max     1267.000000  
ben_dsri    27315
ben_gmi     25638
ben_aqi     20919
ben_sgi     24811


In [30]:
check = (df['rect'] - df['rect_priv_year']) / ((df['at'] + df['at_priv_year']) / 2)
print((check - df['dch_rec']).abs().median())

2.4691161268353135e-10


##### Εδώ διασταυρώνουμε πως λειτουργεί σωστά ο κώδικας με την περσινή χρονιά. Υπολογίστηκε ο δείκτης dch_rec που υπήρχε ήδη στο σύνολο δεδομένων με τον κώδικα που εντοπίζει την προηγούμενη χρονια για να δουμε αν θα βγει το ίδιο αποτέλεσμα και πράγματι η διαφορά μεταξύ των 2 μεταβλητών βγηκε πρακτικά 0, δηλαδή είναι ίδιες.

In [31]:
# Πόσες φορές η "ποιότητα ενεργητικού" βγαίνει αρνητική (λόγω μικτών παγίων)
print("Αρνητικό aq (φέτος):", (aq < 0).sum())
print("Αρνητικό aq_priv_year (πέρσι):", (aq_priv_year < 0).sum())
print("Σύνολο τιμών AQI:", df['ben_aqi'].notna().sum())

Αρνητικό aq (φέτος): 65538
Αρνητικό aq_priv_year (πέρσι): 54579
Σύνολο τιμών AQI: 92062


##### Βρέθηκαν 54579 αρνητικές τιμές της περσινης ποιότητας ενεργητικού λόγω μικτών παγίων άρα ο δείκτης AQI αφαιρέθηκε 

In [32]:
# Αφαίρεση του AQI: δεν υπολογίζεται αξιόπιστα με μικτά πάγια (ppegt)
df = df.drop(columns=['ben_aqi'])

In [33]:
# Αφαίρεση των βοηθητικών στηλών προηγούμενης χρονιάς (χρειάζονταν μόνο για τους δείκτες Beneish)
priv_cols = []
for col in priv_year_cols:
    priv_cols.append(col + '_priv_year')

print("Βοηθητικές στήλες προηγούμενης χρονιάς που θα αφαιρεθούν:", priv_cols)

df = df.drop(columns=priv_cols)
print("Στήλες μετά την αφαίρεση:", df.shape[1])

Βοηθητικές στήλες προηγούμενης χρονιάς που θα αφαιρεθούν: ['rect_priv_year', 'sale_priv_year', 'cogs_priv_year', 'act_priv_year', 'ppegt_priv_year', 'at_priv_year', 'dltt_priv_year', 'lct_priv_year', 'che_priv_year', 'dlc_priv_year', 'txp_priv_year']
Στήλες μετά την αφαίρεση: 54


In [34]:
df = df[df['fyear'] >= 1991].copy()

In [35]:
# Οι 18 δείκτες των τριών scores
f_cols   = ['ch_rsst', 'dch_rec', 'dch_inv', 'soft_assets', 'ch_cs', 'ch_roa', 'issue']   # F-Score (υπήρχαν)
alt_cols = ['alt_x1', 'reoa', 'EBIT', 'alt_x4', 'alt_x5']                                 # Z-Score
ben_cols = ['ben_dsri', 'ben_gmi', 'ben_sgi', 'dpi', 'ben_lvgi', 'ben_tata']   # M-Score

score_cols = f_cols + alt_cols + ben_cols
print("Πλήθος δεικτών:", len(score_cols))

# Πόσες κενές τιμές έχει κάθε δείκτης
print(df[score_cols].isnull().sum())

Πλήθος δεικτών: 18
ch_rsst         3244
dch_rec         3115
dch_inv         3058
soft_assets      391
ch_cs          11268
ch_roa          9253
issue              0
alt_x1           390
reoa             390
EBIT             390
alt_x4           100
alt_x5           390
ben_dsri       22733
ben_gmi        21056
ben_sgi        20229
dpi             5931
ben_lvgi       16445
ben_tata       16251
dtype: int64


#### Παρατηρούμε ότι οι δέικτες του Beneish έχον τα περισσόετερα κενά πράγμα αναμενόμενο καθώς βασίζεται και σε δεδομένα προηγούμενης χρονιάς 

In [36]:
# Χρηματιστηριακή αξία = μετοχές σε κυκλοφορία (εκατ.) × τιμή κλεισίματος ($) -> εκατ. $
df['mkt_val'] = df['csho'] * df['prcc_f']

print(df['mkt_val'].describe())
print("Κενές τιμές:", df['mkt_val'].isnull().sum())
print("Μηδενικές τιμές:", (df['mkt_val'] == 0).sum())

count    1.083990e+05
mean     1.928984e+03
std      1.212768e+04
min      0.000000e+00
25%      1.779777e+01
50%      9.252900e+01
75%      5.369651e+02
max      1.819782e+06
Name: mkt_val, dtype: float64
Κενές τιμές: 0
Μηδενικές τιμές: 15


In [37]:
# Όρια μεγέθους NYSE (Kenneth French Data Library) - ποσά σε εκατ. $
nyse = pd.read_csv("ME_Breakpoints.csv", skiprows=1, header=None,
                   skipfooter=1, engine='python')

# Ονόματα στηλών: ημερομηνία, πλήθος εταιρειών NYSE και τα 20 όρια (5%, 10%, ..., 100%)
nyse_cols = ['date', 'n_firms']
for p in range(5, 105, 5):
    nyse_cols.append('p' + str(p))
nyse.columns = nyse_cols

print(nyse.shape)
print(nyse[['date', 'n_firms', 'p20', 'p40', 'p60', 'p80']].tail(3))

(1209, 22)
        date  n_firms      p20      p40      p60       p80
1206  202606     1079  1224.13  3592.90  7923.39  26898.24
1207  202607     1074  1216.45  3632.39  8264.25  26412.11
1208  202608     1071  1189.07  3657.71  8368.53  25820.10


In [38]:
# Από την ημερομηνία (π.χ. 199112) βγάζουμε το έτος και τον μήνα
nyse['fyear'] = nyse['date'] // 100     # 199112 -> 1991
nyse['month'] = nyse['date'] % 100      # 199112 -> 12

# Κρατάμε μόνο τον Δεκέμβριο των ετών 1991-2008 και τα 4 όρια των πεμπτημορίων
nyse_dec = nyse[(nyse['month'] == 12) & (nyse['fyear'].between(1991, 2008))]
nyse_dec = nyse_dec[['fyear', 'p20', 'p40', 'p60', 'p80']]

print(nyse_dec)
print(nyse_dec.shape)

     fyear     p20      p40      p60      p80
792   1991  102.63   334.01   876.72  2520.15
804   1992  130.85   371.52   965.41  2726.18
816   1993  167.90   430.43  1017.15  2928.71
828   1994  148.85   405.04   939.75  2513.34
840   1995  174.69   474.78  1153.70  3107.33
852   1996  204.38   529.92  1203.22  3443.63
864   1997  259.33   640.35  1404.92  3986.59
876   1998  207.19   537.58  1290.90  4183.24
888   1999  191.21   534.24  1301.13  4276.09
900   2000  189.97   619.68  1583.32  5105.33
912   2001  272.88   757.30  1739.55  5246.23
924   2002  278.16   682.51  1547.52  4446.75
936   2003  457.57  1013.30  2165.36  6138.21
948   2004  631.20  1268.82  2505.37  7339.75
960   2005  623.44  1326.30  2829.94  7950.28
972   2006  747.22  1545.34  3024.02  8595.95
984   2007  565.53  1480.92  3070.68  8820.53
996   2008  288.43   810.96  1866.28  4574.28
(18, 5)


In [39]:
# "Κολλάμε" σε κάθε γραμμή του df τα όρια NYSE της χρονιάς της 
rows_before = len(df)
df = df.merge(nyse_dec, on='fyear', how='left')

print("Γραμμές πριν:", rows_before, "| μετά:", len(df))
print("Γραμμές χωρίς όρια:", df['p20'].isnull().sum())
print(df[['gvkey', 'fyear', 'mkt_val', 'p20', 'p40', 'p60', 'p80']].head())

Γραμμές πριν: 108399 | μετά: 108399
Γραμμές χωρίς όρια: 0
   gvkey  fyear     mkt_val     p20      p40      p60      p80
0   1004   1991  204.699625  102.63   334.01   876.72  2520.15
1   1004   1997  732.424500  259.33   640.35  1404.92  3986.59
2   1004   2001  364.592800  272.88   757.30  1739.55  5246.23
3   1004   2002  143.329500  278.16   682.51  1547.52  4446.75
4   1004   2003  308.907100  457.57  1013.30  2165.36  6138.21


In [40]:
# Κατηγορία μεγέθους 1-5 (πεμπτημόρια NYSE):
# ξεκινάμε από 1 και προσθέτουμε 1 για κάθε όριο που ξεπερνά η εταιρεία
df['size_nyse'] = (1
                   + (df['mkt_val'] >= df['p20']).astype(int)
                   + (df['mkt_val'] >= df['p40']).astype(int)
                   + (df['mkt_val'] >= df['p60']).astype(int)
                   + (df['mkt_val'] >= df['p80']).astype(int))

# Έλεγχος στις πρώτες γραμμές
print(df[['fyear', 'mkt_val', 'p20', 'p40', 'p60', 'p80', 'size_nyse']].head())

# Πόσες εταιρείες σε κάθε κατηγορία (πλήθος και %)
print(df['size_nyse'].value_counts().sort_index())
print((df['size_nyse'].value_counts(normalize=True).sort_index() * 100).round(1))

   fyear     mkt_val     p20      p40      p60      p80  size_nyse
0   1991  204.699625  102.63   334.01   876.72  2520.15          2
1   1997  732.424500  259.33   640.35  1404.92  3986.59          3
2   2001  364.592800  272.88   757.30  1739.55  5246.23          2
3   2002  143.329500  278.16   682.51  1547.52  4446.75          1
4   2003  308.907100  457.57  1013.30  2165.36  6138.21          1
size_nyse
1    71295
2    13526
3     8874
4     7508
5     7196
Name: count, dtype: int64
size_nyse
1    65.8
2    12.5
3     8.2
4     6.9
5     6.6
Name: proportion, dtype: float64


In [41]:
# Έλεγχος αναλογιών: train (1991-2001) έναντι test (2003-2008)
train_part = df[df['fyear'] <= 2001]
test_part = df[df['fyear'] >= 2003]

size_share = pd.DataFrame({
    'train_%': train_part['size_nyse'].value_counts(normalize=True).sort_index() * 100,
    'test_%':  test_part['size_nyse'].value_counts(normalize=True).sort_index() * 100,
})
print(size_share.round(1))

# Απάτες και ποσοστό απάτης ανά κατηγορία μεγέθους (όλη η περίοδος 1991-2008)
size_fraud = df.groupby('size_nyse')['misstate'].agg(['count', 'sum', 'mean'])
size_fraud.columns = ['Γραμμές', 'Απάτες', 'Ποσοστό απάτης (%)']
size_fraud['Ποσοστό απάτης (%)'] = (size_fraud['Ποσοστό απάτης (%)'] * 100).round(2)
print(size_fraud)

           train_%  test_%
size_nyse                 
1             64.2    68.0
2             13.7    10.6
3              8.7     7.4
4              7.1     6.8
5              6.3     7.3
           Γραμμές  Απάτες  Ποσοστό απάτης (%)
size_nyse                                     
1            71295     326                0.46
2            13526     157                1.16
3             8874     103                1.16
4             7508      90                1.20
5             7196     161                2.24


In [42]:
# Αφαίρεση των βοηθητικών στηλών με τα όρια NYSE (χρειάζονταν μόνο για την κατηγορία)
df = df.drop(columns=['p20', 'p40', 'p60', 'p80'])
print("Στήλες:", df.shape[1])

Στήλες: 56


In [43]:
# Προφίλ κάθε κατηγορίας μεγέθους - διάμεσοι, γιατί τα ποσά είναι πολύ ασύμμετρα
profile = pd.DataFrame({
    'size_nyse':               df['size_nyse'],
    'Πωλήσεις':                df['sale'],                          # κύκλος εργασιών (εκατ. $)
    'Ενεργητικό':              df['at'],                            # εκατ. $
    'Μικτά ενσώματα πάγια':    df['ppegt'],                         # εκατ. $
    'Ίδια κεφάλαια':           df['ceq'],                           # κοινά ίδια κεφάλαια (εκατ. $)
    'EBIT (εκατ. $)':          df['ib'] + df['xint'] + df['txt'],   # προσέγγιση: κέρδη προ έκτακτων + τόκοι + φόροι
    'EBIT / Ενεργητικό':       df['EBIT'],                          # έτοιμη στήλη του dataset
    'Μόχλευση':                df['lt'] / df['at'],                 # σύνολο υποχρεώσεων / σύνολο ενεργητικού
    'Ρευστότητα':              df['act'] / df['lct'],               # κυκλοφορούν ενεργητικό / βραχυπρόθεσμες υποχρεώσεις
})
profile = profile.replace([np.inf, -np.inf], np.nan)   # διαιρέσεις με το 0 -> κενή τιμή

size_profile = profile.groupby('size_nyse').median().round(2)
print(size_profile)

           Πωλήσεις  Ενεργητικό  Μικτά ενσώματα πάγια  Ίδια κεφάλαια  \
size_nyse                                                              
1             26.63       32.85                 11.04          14.15   
2            303.70      352.95                140.01         163.05   
3            730.00      886.09                431.81         375.30   
4           1773.87     2379.06               1296.58         938.46   
5           8000.18    10890.00               6650.67        4073.05   

           EBIT (εκατ. $)  EBIT / Ενεργητικό  Μόχλευση  Ρευστότητα  
size_nyse                                                           
1                   -0.03              -0.00      0.48        1.80  
2                   26.16               0.08      0.47        2.10  
3                   77.60               0.09      0.52        1.83  
4                  211.11               0.10      0.56        1.59  
5                  998.15               0.10      0.58        1.28  


In [44]:
# Αποθήκευση του df με όλες τις νέες μεταβλητές - το διαβάζει το 03_models
df.to_pickle("df_features.pkl")
print("Αποθηκεύτηκε:", df.shape)

Αποθηκεύτηκε: (108399, 56)
